# MAMC Pipeline Diagnostics Walkthrough
This notebook provides a quick interactive demonstration of the **MAMC** framework. It loads the target configuration registry and processes timing anomaly models across three separated diagnostic suites.

### Dependencies
Ensure you have the core files (`mamc.py` and `target_params.py`) in the same directory as this notebook.

In [1]:
import pandas as pd
from target_params import TARGETS
from mamc import ApplegateMechanism, LanzaMechanism, AzimuthalDynamoWave

# Set global formatting configurations for scannable output rows
pd.options.display.float_format = "{:.3e}".format

In [2]:
# Initialize clean, isolated data structures for reporting arrays
applegate_records = []
lanza_records = []
adw_records = []

for name, sys in TARGETS.items():
    # 1. Evaluate Applegate Analytical Thresholds
    r_thin = ApplegateMechanism.thin_shell_ratio(
        M_sec=sys.M_sec, R_sec=sys.R_sec, T_sec=sys.T_sec, a_bin=sys.a_bin, 
        P_bin=sys.P_bin_days, P_mod_yr=sys.P_mod_yr, A_sec=sys.A_sec
    )
    r_const = ApplegateMechanism.constant_density_ratio(
        M_sec=sys.M_sec, R_sec=sys.R_sec, L_sec=sys.L_sec, a_bin=sys.a_bin, 
        P_mod_yr=sys.P_mod_yr, A_sec=sys.A_sec
    )
    r_two, app_A = ApplegateMechanism.two_zone_ratio(
        M_sec=sys.M_sec, R_sec=sys.R_sec, L_sec=sys.L_sec, a_bin=sys.a_bin, 
        P_bin=sys.P_bin_days, dP_over_P=sys.dP_over_Pbin, P_mod_yr=sys.P_mod_yr
    )
    applegate_records.append({
        "System": sys.name, "Applegate A": app_A, 
        "Thin-Shell": r_thin, "Const-Density": r_const, "Two-Zone": r_two
    })

    # 2. Evaluate Lanza Spin-Orbit Shear Metrics
    l_yr, l_mod, dOm, _ = LanzaMechanism.analyze(
        P_bin=sys.P_bin_days, P_mod_yr=sys.P_mod_yr, A_sec=sys.A_sec,
        M_pri=sys.M_pri, M_sec=sys.M_sec, R_sec=sys.R_sec, T_sec=sys.T_sec, 
        L_sec=sys.L_sec, a_bin=sys.a_bin
    )
    lanza_records.append({
        "System": sys.name, "Observed A (s)": sys.A_sec, 
        "dOmega/Omega": dOm, "dE/L (1 yr)": l_yr, "dE/L (Pmod)": l_mod
    })

    # 3. Evaluate Non-Axisymmetric Quadrupole Changes (ADW Model)
    dQ_xx, theoretical_OC, v_lin, v_ratio = AzimuthalDynamoWave.analyze(
        M_sec=sys.M_sec, R_sec=sys.R_sec, a_bin=sys.a_bin, 
        P_bin=sys.P_bin_days, P_mod_yr=sys.P_mod_yr, f_scale=1.0
    )
    adw_records.append({
        "System": sys.name, "Observed A (s)": sys.A_sec, 
        "V_eq (km/s)": v_lin, "Omega/Omega_sun": v_ratio, 
        "Delta Q_xx": dQ_xx, "Expected O-C (s)": theoretical_OC
    })

## Suite 1: Applegate Mechanism Energy Budgets (Völschow et al. 2016)
This section checks the dimensionless structural coupling parameter ($A$) and evaluates the required relative energy efficiency budget ratio ($\Delta E / E_{\text{sec}}$) across various geometric configurations.

In [3]:
df_app = pd.DataFrame(applegate_records)
df_app.style.format({
    "Applegate A": "{:.3f}", "Thin-Shell": "{:.2f}", 
    "Const-Density": "{:.2f}", "Two-Zone": "{:.2f}"
})

,System,Applegate A,Thin-Shell,Const-Density,Two-Zone
0,NN Ser,0.159,3.30,1116.87,62.72
1,HW Vir,0.361,5.99,756.17,110.25
2,QS Vir,0.012,0.04,179.90,0.71
3,DD CrB,2.207,0.60,715.03,-1.00
4,NY Vir,0.624,0.55,1089.71,7656.24


### Diagnostic Instructions:
* **Applegate A**: Must be $\le 1.0$. If $A > 1$, a physical core-shell solution cannot exist (returns `-1`).
* **$\Delta E / E_{\text{sec}} \ll 1$**: Energetically highly feasible (e.g., QS Vir).
* **$\Delta E / E_{\text{sec}} \gg 1$**: Energetically impossible; requires more energy than the secondary star produces over the cycle (e.g., NN Ser, HW Vir).

## Suite 2: Lanza Spin-Orbit Coupling Efficiency (Lanza 2020)
This suite evaluates the mechanical kinetic energy work done by cyclic convective envelope speed modulations ($\Delta \Omega / \Omega$) against the target's absolute integrated radiant output.

In [4]:
df_lan = pd.DataFrame(lanza_records)
df_lan.style.format({
    "Observed A (s)": "{:.2f}", "dOmega/Omega": "{:.3e}", 
    "dE/L (1 yr)": "{:.2f}", "dE/L (Pmod)": "{:.2f}"
})

,System,Observed A (s),dOmega/Omega,dE/L (1 yr),dE/L (Pmod)
0,NN Ser,27.65,-3.857e-05,322.40,20.82
1,HW Vir,563.00,-1.269e-04,1132.41,20.59
2,QS Vir,43.00,-9.900e-06,190.08,11.19
3,DD CrB,5.10,-7.752e-06,118.70,8.93
4,NY Vir,17.55,-1.008e-05,684.07,30.61


### Diagnostic Instructions:
* Focus on the **`dE/L (Pmod)`** column, which reveals the strict percentage of integrated radiant luminosity that must be continuously converted into internal mechanical shear.
* Ratios $\ge 1.0$ indicate that the mechanism is completely disproven for that target star.

## Suite 3: Azimuthal Dynamo Wave Amplitudes (Navarrete et al. 2026)

#### A work in progress!

This script sets a baseline dynamo scaling factor ($f_{\text{scale}} = 1.0$) to test if non-axisymmetric polar mass deformations can directly generate the observed O-C variation depths.

In [6]:
df_adw = pd.DataFrame(adw_records)
df_adw.style.format({
    "Observed A (s)": "{:.2f}", "V_eq (km/s)": "{:.2f}", 
    "Omega/Omega_sun": "{:.2f}", "Delta Q_xx": "{:.2e}", "Expected O-C (s)": "{:.2f}"
})

,System,Observed A (s),V_eq (km/s),Omega/Omega_sun,Delta Q_xx,Expected O-C (s)
0,NN Ser,27.65,57.99,192.69,-2.70e+47,-202.55
1,HW Vir,563.00,75.67,214.10,-2.70e+47,-663.43
2,QS Vir,43.00,140.72,165.89,-2.70e+47,-31.03
3,DD CrB,5.10,50.63,154.85,-2.70e+47,-127.37
4,NY Vir,17.55,77.63,247.98,-2.70e+47,-367.34


### Diagnostic Instructions:
* **Negative $\Delta Q_{xx}$**: A real physical trait of rapid rotators showing a geometry shift from oblate to prolate configuration shapes.
* **Expected vs. Observed**: If the `Expected O-C` matches or beats the `Observed A` row baseline even under a minimum scaling footprint ($f = 1.0$), the ADW model stands as a highly robust mechanism.

# Citation:

Please cite <a href="https://ui.adsabs.harvard.edu/abs/2026MNRAS.547ag290B/abstract">DD CrB work from our group</a> if you use this code for the computation of energy requirements for Applegate mechanism. 

You can use the computations for the ADW mechanism at your own risk. Our paper on DW UMa is in preparation for the moment, some of the results of which are based on the computation with this mechanism. But that part of of the code is a work in progress!